# sre-gym — SFT warm-start on the combined teacher seed

Open in Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dakshdoesdev/sre-enginnerllm/blob/main/train/sanity_run.ipynb)

Purpose: SFT on the 200-sample mixed-teacher corpus (Claude + Llama-70B Fireworks + Llama-70B Groq). The resulting LoRA adapter becomes the GRPO starting point in `grpo_run.ipynb`.

What a successful run looks like:
1. Cell 6 — `HF_TOKEN` lands from Colab Secrets, deps install OK
2. Cell 8 — `Qwen3.5-4B-Instruct` (or Qwen3-4B fallback) loads in 4-bit via Unsloth on A100 40GB
3. Cell 10 — auto-downloads `seed_combined.jsonl` from GitHub, loads ~200 chat-formatted samples
4. Cell 12 — 500 LoRA SFT steps, loss drops ~2.0 → < 0.5, checkpoints push to `dakshdoesdev/sre-gym-qwen35-4b-sft` every save
5. Cell 14 — final adapter pushed; inference cell emits valid JSON action

**Before clicking Run all**:
- Add `HFTOKEN` to Colab Secrets (left sidebar 🔑 panel, toggle "Notebook access" on)
- Optionally add `WANDB_API_KEY` for the live training-curve dashboard
- Runtime → Change runtime type → A100 GPU

No file uploads needed — the seed dataset auto-downloads from the public GitHub repo.

## 0. Colab runtime sanity

In [ ]:
!nvidia-smi
!python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

## 1. Install deps

In [ ]:
%%bash
# Unsloth's Colab install idiom (handles torch/xformers version pinning):
pip install -q --upgrade pip
pip install -q "unsloth[colab-new]>=2025.12,<2026.06"
pip install -q "unsloth_zoo>=2025.12,<2026.06"
pip install -q "trl>=0.12,<0.16" "transformers>=4.48,<4.60" "peft>=0.14,<0.20" "accelerate>=1.2,<2.0"
pip install -q "datasets>=3.0,<4.0" "wandb>=0.18,<1.0" "bitsandbytes>=0.45" httpx

## 2. Config

If Qwen3.5 4B fails to load, swap `MODEL_NAME` to the Qwen3 4B fallback — no other change needed.

In [ ]:
import os

# --- Colab secrets bridge ------------------------------------------------
# Colab stores secrets in `google.colab.userdata` (left-sidebar 🔑 panel),
# NOT the process env. Pull HFTOKEN (or HF_TOKEN) and a wandb key across
# so trainer libs (transformers, wandb) see them as env vars.
try:
    from google.colab import userdata  # type: ignore
    for _src in ("HF_TOKEN", "HFTOKEN", "HUGGINGFACE_TOKEN"):
        try:
            v = userdata.get(_src)
            if v:
                os.environ["HF_TOKEN"] = v
                break
        except Exception:
            pass
    for _src in ("WANDB_API_KEY", "WANDB", "wandb"):
        try:
            v = userdata.get(_src)
            if v:
                os.environ["WANDB_API_KEY"] = v
                break
        except Exception:
            pass
except ImportError:
    pass  # running outside Colab — keep existing env vars

if "HF_TOKEN" not in os.environ:
    raise RuntimeError(
        "HF_TOKEN not set. In Colab: left sidebar 🔑 -> add a secret named "
        "HFTOKEN (or HF_TOKEN), toggle 'Notebook access' on, re-run this cell."
    )
# -------------------------------------------------------------------------

# Primary target.
MODEL_NAME = "unsloth/Qwen3.5-4B-Instruct-bnb-4bit"
# Fallback if Unsloth can't load Qwen3.5 on Colab tonight.
FALLBACK_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 4096
LORA_R = 32
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
NUM_STEPS = 500          # ~200 usable samples * ~10 epochs at BATCH_SIZE=2, GRAD_ACCUM=4
BATCH_SIZE = 2
GRAD_ACCUM = 4
OUT_DIR = "/content/sanity_ckpt"
# Combined seed: Claude Opus + Llama-70B (Fireworks) + Llama-70B (Groq)
# Upload train/data/seed_combined.jsonl to /content/ before running.
SEED_JSONL_PATH = "/content/seed_combined.jsonl"
HUB_MODEL_ID = "dakshdoesdev/sre-gym-qwen35-4b-sft"

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "sre-gym-sft")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", "qwen35-4b-sft-combined-500")

# Auto-toggle wandb mode based on key presence.
os.environ.setdefault(
    "WANDB_MODE",
    "online" if os.environ.get("WANDB_API_KEY") else "offline",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

print(f"HF_TOKEN set:       {'yes' if os.environ.get('HF_TOKEN') else 'NO'}")
print(f"WANDB_API_KEY set:  {'yes (online)' if os.environ.get('WANDB_API_KEY') else 'no (offline mode)'}")
print(f"Primary model:      {MODEL_NAME}")
print(f"Seed dataset:       {SEED_JSONL_PATH}")
print(f"Training {NUM_STEPS} steps -> hub: {HUB_MODEL_ID}")

## 3. Load model via Unsloth (with fallback)

In [ ]:
from unsloth import FastLanguageModel
import torch

model = None
tokenizer = None
errors = []

for candidate in (MODEL_NAME, FALLBACK_MODEL_NAME):
    try:
        print(f"Attempting to load: {candidate}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=candidate,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,  # let Unsloth pick
            load_in_4bit=True,
        )
        MODEL_NAME = candidate
        print(f"Loaded {candidate} ok")
        break
    except Exception as exc:
        errors.append((candidate, repr(exc)))
        print(f"Load failed for {candidate}: {exc}")

if model is None:
    raise RuntimeError(
        "Both Qwen3.5 4B and Qwen3 4B failed to load via Unsloth. "
        "Investigate Unsloth version mismatch before Friday. Errors: " + str(errors)
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Combined teacher seed dataset (200 usable samples, 21 episodes, 3 teachers)

Cell auto-downloads `seed_combined.jsonl` from the GitHub repo on first run, then flattens per-step `(prompt, response_text)` pairs into chat-formatted training text.

| Teacher | Episodes | Resolved | Mean score | Role |
|---|---|---|---|---|
| Claude Opus 4.7 | 6 | 6/6 | 0.769 | Expert demos |
| Llama-3.3-70B (Fireworks) | 4 | 3/4 | 0.725 | Solid agent |
| Llama-3.3-70B (Groq) | 11 | 5/11 | 0.421 | Noisy realistic agent |

Filter: skip samples where `len(prompt) < 50` (4 Claude rollback steps lost their prior observation to a chained-call logging bug). Net: 200 trainable (prompt, action) pairs covering all 6 scenario templates.

In [ ]:
import json, os, urllib.request
from datasets import Dataset

# Auto-download the seed dataset from GitHub if not already present.
SEED_GITHUB_URL = "https://raw.githubusercontent.com/dakshdoesdev/sre-enginnerllm/main/train/data/seed_combined.jsonl"
if not os.path.exists(SEED_JSONL_PATH):
    print(f"downloading {SEED_GITHUB_URL} -> {SEED_JSONL_PATH}")
    urllib.request.urlretrieve(SEED_GITHUB_URL, SEED_JSONL_PATH)

SYSTEM = (
    "You are an SRE agent operating inside the sre-gym environment. "
    "On each turn, emit exactly one UnifiedIncidentAction JSON object — no surrounding prose, "
    "no code fences, no commentary. Valid action_types: query_logs, query_metrics, "
    "query_dependencies, query_deploys, rollback_deploy, restart_service, isolate_service, "
    "run_check, submit_hypothesis, escalate, declare_resolved."
)


def _load_seed_samples(path: str):
    """Flatten canonical trajectory JSONL into (prompt, response_text) pairs.

    Skips steps whose prompt is empty (4 Claude rollback steps lost their
    prior observation to a chained-call logging bug — 200 of 204 raw steps
    are clean, plenty for format-anchoring SFT).
    """
    samples = []
    held_out_prompt = None
    skipped = 0
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ep = json.loads(line)
            for step in ep["trajectory"]:
                prompt = step.get("prompt") or ""
                response = step.get("response_text") or ""
                if len(prompt) < 50 or not response:
                    skipped += 1
                    continue
                samples.append((prompt, response))
            if held_out_prompt is None and ep["trajectory"]:
                first = ep["trajectory"][0]
                if first.get("prompt") and len(first["prompt"]) >= 50:
                    held_out_prompt = first["prompt"]
    print(f"loaded {len(samples)} training samples, skipped {skipped} incomplete")
    return samples, held_out_prompt


samples, HELD_OUT_PROMPT = _load_seed_samples(SEED_JSONL_PATH)
if not samples:
    raise RuntimeError(f"No samples found in {SEED_JSONL_PATH}.")


def _format(example):
    prompt, action = example
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": action},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}


raw = [_format(ex) for ex in samples]
dataset = Dataset.from_list(raw)
print(f"dataset: {len(dataset)} rows")
print("sample text (first 600 chars):")
print(dataset[0]["text"][:600])

## 5. SFT training — 500 steps

~10 epochs over the 200 samples at `batch_size=2, grad_accum=4`. Pushes adapter checkpoints to `dakshdoesdev/sre-gym-qwen35-4b-sft` on every save (so a Colab disconnect doesn't lose work). On A100 40GB this is ~20 minutes wall-clock.

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    max_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    # Push checkpoints to the Hub as we save — survives Colab disconnects.
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="every_save",
    hub_private_repo=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=cfg,
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. Save LoRA adapter + sanity-check inference

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# Final push (in addition to the per-checkpoint pushes during training).
trainer.push_to_hub()
print(f"Pushed final adapter to https://huggingface.co/{HUB_MODEL_ID}")

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# Use the held-out prompt captured during dataset load (first step of first episode).
# Should emit a valid UnifiedIncidentAction JSON object.
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": HELD_OUT_PROMPT},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=96, temperature=0.0, do_sample=False)
generated = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
print("GENERATED ACTION:")
print(generated)

import json as _json
try:
    parsed = _json.loads(generated.strip())
    assert "action_type" in parsed
    print(f"\nFORMAT OK: action_type={parsed['action_type']}")
except Exception as exc:
    print(f"\nFORMAT WARNING: {exc}. Check that SFT learned the JSON shape.")

## Verification checklist

- [ ] Cell 3 loaded a model without OOM or import errors
- [ ] Cell 4 produced 39 chat-formatted rows (no tokenizer errors, no missing JSONL file)
- [ ] Cell 5 ran 500 steps, wandb logged a decreasing loss curve (~2.0 → < 0.5)
- [ ] Cell 6 generated format-valid JSON with `action_type` field
- [ ] `/content/sanity_ckpt/adapter_model.safetensors` exists

If any box is unchecked, debug before moving to GRPO — the GRPO run will burn 4–6 hours of A100 time and any upstream format bug amplifies there.

## Next step after this notebook passes

1. Upload the LoRA adapter to `dakshdoesdev/sre-gym-qwen35-4b-sft` (use `huggingface_hub.upload_folder` or `hf upload`).
2. Clone `Gen-Verse/OpenClaw-RL` in a separate venv.
3. Apply the one-line import patch documented in `openclaw_integration/README.md`.
4. Boot `openclaw_integration/pool_server.py` on port 8100.
5. Launch OpenClaw-RL's GRPO script with `ENV_SERVER_URL=http://127.0.0.1:8100` and starting weights = this SFT checkpoint.
6. Target 500 GRPO steps.